# AutoML Benchmark Analysis

**Comparative Evaluation of AutoML Frameworks for Tabular Data in Resource-Constrained Environments**

Reads `results/logs.csv` (105 rows: 5 frameworks x 3 tiers x 7 datasets) and produces:

1. Master summary table — mean F1, time, RAM per framework per tier
2. Pareto frontier plot — accuracy vs. memory tradeoff per tier
3. Degradation curves — F1 drop as resources shrink per framework
4. Failure heatmap — completion status across all 105 runs
5. Recommendation matrix — which framework for which context


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid")
FIGURES = Path("../figures")
FIGURES.mkdir(exist_ok=True)

TIER_ORDER = ["constrained", "moderate", "unconstrained"]
FRAMEWORKS = ["autogluon", "pycaret", "flaml", "autosklearn", "h2o"]

logs = pd.read_csv("../results/logs.csv")
logs["tier"] = pd.Categorical(logs["tier"], categories=TIER_ORDER, ordered=True)
logs["completed"] = logs["status"].eq("completed")
print(f"{len(logs)} runs loaded, {logs['completed'].sum()} completed")
logs.head()

## 1. Master summary table

Mean F1 (classification), RMSE (regression), training time, and peak RAM per framework per tier — completed runs only.

In [ ]:
completed = logs[logs["completed"]]

summary = (
    completed.groupby(["framework", "tier"], observed=True)
    .agg(
        mean_f1=("f1", "mean"),
        mean_rmse=("rmse", "mean"),
        mean_time_sec=("time_sec", "mean"),
        mean_peak_ram_mb=("peak_ram_mb", "mean"),
        mean_cpu_pct=("cpu_pct", "mean"),
        runs_completed=("completed", "sum"),
    )
    .round(3)
)
summary

## 2. Pareto frontier: accuracy vs. memory per tier

Each point is one framework's mean weighted F1 (classification datasets) against its mean peak RAM within a tier. Up and to the left is better.

In [ ]:
clf = completed[completed["f1"].notna()]
agg = (
    clf.groupby(["framework", "tier"], observed=True)
    .agg(f1=("f1", "mean"), ram=("peak_ram_mb", "mean"))
    .reset_index()
)

fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=True)
for ax, tier in zip(axes, TIER_ORDER):
    sub = agg[agg["tier"] == tier]
    ax.scatter(sub["ram"], sub["f1"], s=80)
    for _, r in sub.iterrows():
        ax.annotate(r["framework"], (r["ram"], r["f1"]),
                    textcoords="offset points", xytext=(6, 4))
    # Pareto frontier: sort by RAM, keep points with a new best F1
    front = sub.sort_values("ram")
    best, xs, ys = -np.inf, [], []
    for _, r in front.iterrows():
        if r["f1"] > best:
            best = r["f1"]; xs.append(r["ram"]); ys.append(r["f1"])
    ax.step(xs, ys, where="post", alpha=0.5, color="grey")
    ax.set_title(f"{tier} tier")
    ax.set_xlabel("Mean peak RAM (MB)")
axes[0].set_ylabel("Mean weighted F1")
fig.suptitle("Accuracy vs. memory tradeoff per tier")
fig.tight_layout()
fig.savefig(FIGURES / "pareto_plot.png", dpi=200, bbox_inches="tight")
plt.show()

## 3. Degradation curves

How much does each framework's mean F1 drop as resources shrink from the unconstrained tier down to the constrained tier?

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5.5))
for fw in FRAMEWORKS:
    sub = (
        clf[clf["framework"] == fw]
        .groupby("tier", observed=False)["f1"].mean()
        .reindex(TIER_ORDER)
    )
    ax.plot(TIER_ORDER, sub.values, marker="o", label=fw)
ax.set_xlabel("Resource tier")
ax.set_ylabel("Mean weighted F1 (classification datasets)")
ax.set_title("F1 degradation as resources shrink")
ax.legend(title="Framework")
fig.tight_layout()
fig.savefig(FIGURES / "degradation_curves.png", dpi=200, bbox_inches="tight")
plt.show()

## 4. Failure heatmap

Completion status for every (framework, dataset) cell in each tier:
`2 = completed`, `1 = timeout`, `0 = failed`.

In [ ]:
status_code = logs["status"].map(
    lambda s: 2 if s == "completed" else (1 if s == "timeout" else 0)
)
logs["status_code"] = status_code

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, tier in zip(axes, TIER_ORDER):
    pivot = (
        logs[logs["tier"] == tier]
        .pivot_table(index="framework", columns="dataset",
                     values="status_code", aggfunc="min")
        .reindex(FRAMEWORKS)
    )
    sns.heatmap(pivot, ax=ax, cmap="RdYlGn", vmin=0, vmax=2,
                annot=True, cbar=False, linewidths=0.5)
    ax.set_title(f"{tier} tier")
fig.suptitle("Completion status (2=completed, 1=timeout, 0=failed)")
fig.tight_layout()
fig.savefig(FIGURES / "failure_heatmap.png", dpi=200, bbox_inches="tight")
plt.show()

## 5. Recommendation matrix

For each tier: the best framework by mean F1, the most memory-efficient among near-best performers, and the completion rate — the practical selection guide for each hardware context.

In [ ]:
rows = []
for tier in TIER_ORDER:
    tsub = agg[agg["tier"] == tier]
    tier_logs = logs[logs["tier"] == tier]
    if tsub.empty:
        continue
    best = tsub.loc[tsub["f1"].idxmax()]
    # near-best = within 2 F1 points of the top performer
    near = tsub[tsub["f1"] >= best["f1"] - 0.02]
    efficient = near.loc[near["ram"].idxmin()]
    rows.append({
        "tier": tier,
        "best_f1_framework": best["framework"],
        "best_f1": round(best["f1"], 3),
        "most_efficient_near_best": efficient["framework"],
        "its_peak_ram_mb": round(efficient["ram"], 0),
        "tier_completion_rate": round(tier_logs["completed"].mean(), 2),
    })

recommendation = pd.DataFrame(rows)
recommendation

---

*All experiments seeded at 42; resource tiers enforced via Docker. See `README.md` for the full experiment design.*